In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# ============================================================
# MIMIC-IV SAE FEATURE ENGINEERING
# ============================================================

ROOT = Path("/Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning")
MIMIC = ROOT / "data" / "raw" / "mimic-iv"
HOSP = MIMIC / "hosp"
ICU = MIMIC / "icu"

OUT = ROOT / "sae_mimiciv_v2" / "data"
OUT.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("MIMIC-IV:", MIMIC)

# ============================================================
# 1. LOAD SAE ICU COHORT
# ============================================================

diagnoses = pd.read_csv(HOSP / "diagnoses_icd.csv.gz")
d_icd = pd.read_csv(HOSP / "d_icd_diagnoses.csv.gz")
icustays = pd.read_csv(ICU / "icustays.csv.gz")

# Find sepsis codes
text_cols = [c for c in ["long_title", "short_title"] if c in d_icd.columns]

sepsis_mask = False
for c in text_cols:
    sepsis_mask = sepsis_mask | d_icd[c].astype(str).str.contains(
        "sepsis|septic",
        case=False,
        na=False,
        regex=True
    )

sepsis_codes = d_icd.loc[
    sepsis_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

# Find encephalopathy codes
enceph_mask = False
for c in text_cols:
    enceph_mask = enceph_mask | d_icd[c].astype(str).str.contains(
        "encephal",
        case=False,
        na=False,
        regex=True
    )

enceph_codes = d_icd.loc[
    enceph_mask,
    ["icd_code", "icd_version"]
].drop_duplicates()

# Sepsis admissions
sepsis_dx = diagnoses.merge(
    sepsis_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

# Encephalopathy admissions
enceph_dx = diagnoses.merge(
    enceph_codes,
    on=["icd_code", "icd_version"],
    how="inner"
)

# SAE = sepsis + encephalopathy
sae_hadm = set(sepsis_dx["hadm_id"]).intersection(
    set(enceph_dx["hadm_id"])
)

sae_icu = icustays[
    icustays["hadm_id"].isin(sae_hadm)
].copy()

sae_icu["intime"] = pd.to_datetime(sae_icu["intime"])
sae_icu["outtime"] = pd.to_datetime(sae_icu["outtime"])

print("=" * 60)
print("SAE COHORT")
print("=" * 60)
print("SAE admissions:", len(sae_hadm))
print("SAE patients:", sae_icu["subject_id"].nunique())
print("SAE ICU stays:", sae_icu["stay_id"].nunique())
print("=" * 60)

# ============================================================
# 2. LOAD ITEM DICTIONARY
# ============================================================

d_items = pd.read_csv(ICU / "d_items.csv.gz")

print("\nICU item dictionary:", d_items.shape)

# Search relevant clinical variables
patterns = {
    "heart_rate": r"heart rate",
    "resp_rate": r"respiratory rate",
    "spo2": r"oxygen saturation|spo2",
    "map": r"mean arterial pressure|\bmap\b",
    "temperature": r"temperature",
    "gcs_total": r"gcs total|glasgow coma scale total",
    "gcs_eye": r"gcs - eye|gcs eye",
    "gcs_motor": r"gcs - motor|gcs motor",
    "gcs_verbal": r"gcs - verbal|gcs verbal",
    "rass": r"richmond agitation|rass",
}

item_groups = {}

for name, pattern in patterns.items():

    mask = (
        d_items["label"]
        .astype(str)
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    )

    matches = d_items.loc[mask].copy()

    item_groups[name] = set(matches["itemid"].astype(int))

    print(f"\n{name}: {len(matches)} matching items")

    if len(matches) > 0:
        print(
            matches[
                ["itemid", "label"]
            ].head(10).to_string(index=False)
        )

# ============================================================
# 3. LOAD LAB DICTIONARY
# ============================================================

d_labitems = pd.read_csv(
    HOSP / "d_labitems.csv.gz"
)

print("\nLab dictionary:", d_labitems.shape)

lab_patterns = {
    "lactate": r"lactate",
    "wbc": r"white blood cell|wbc",
    "creatinine": r"creatinine",
    "bilirubin": r"bilirubin",
    "platelets": r"platelet",
    "bun": r"\bbun\b|urea nitrogen",
    "sodium": r"sodium",
    "glucose": r"glucose",
    "ph": r"^ph$|blood ph|arterial ph",
}

lab_groups = {}

for name, pattern in lab_patterns.items():

    mask = (
        d_labitems["label"]
        .astype(str)
        .str.contains(
            pattern,
            case=False,
            na=False,
            regex=True
        )
    )

    matches = d_labitems.loc[mask].copy()

    lab_groups[name] = set(matches["itemid"].astype(int))

    print(f"\n{name}: {len(matches)} matching lab items")

    if len(matches) > 0:
        print(
            matches[
                ["itemid", "label"]
            ].head(10).to_string(index=False)
        )

# ============================================================
# 4. LOAD CHART EVENTS ONLY FOR SAE ICU STAYS
# ============================================================

stay_ids = set(sae_icu["stay_id"].astype(int))

needed_chart_items = set()

for ids in item_groups.values():
    needed_chart_items.update(ids)

print(
    "\nChart item IDs required:",
    len(needed_chart_items)
)

chart_parts = []

# Read in chunks so we do not load the entire table unnecessarily
for chunk in pd.read_csv(
    ICU / "chartevents.csv.gz",
    compression="gzip",
    chunksize=500_000
):

    filtered = chunk[
        chunk["stay_id"].isin(stay_ids)
        &
        chunk["itemid"].isin(needed_chart_items)
    ].copy()

    if len(filtered) > 0:
        chart_parts.append(filtered)

if chart_parts:
    chart = pd.concat(
        chart_parts,
        ignore_index=True
    )
else:
    chart = pd.DataFrame()

print("\nFiltered chart events:", chart.shape)

# ============================================================
# 5. MAP CHART ITEMS TO CLINICAL FEATURES
# ============================================================

if len(chart) > 0:

    item_to_feature = {}

    for feature, ids in item_groups.items():
        for itemid in ids:
            item_to_feature[itemid] = feature

    chart["feature"] = chart["itemid"].map(
        item_to_feature
    )

    chart["charttime"] = pd.to_datetime(
        chart["charttime"]
    )

    chart["valuenum"] = pd.to_numeric(
        chart["valuenum"],
        errors="coerce"
    )

    chart = chart.dropna(
        subset=["valuenum", "charttime"]
    )

    chart = chart[
        ["subject_id",
         "hadm_id",
         "stay_id",
         "charttime",
         "feature",
         "valuenum"]
    ]

    # If multiple measurements occur in an hour,
    # use the median measurement.
    chart["hour_time"] = (
        chart["charttime"]
        .dt.floor("h")
    )

    chart_hourly = (
        chart
        .groupby(
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "hour_time",
                "feature"
            ],
            as_index=False
        )["valuenum"]
        .median()
    )

    chart_hourly = chart_hourly.pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="first"
    ).reset_index()

else:

    chart_hourly = pd.DataFrame()

print(
    "\nHourly chart feature table:",
    chart_hourly.shape
)

# ============================================================
# 6. LOAD LAB EVENTS FOR SAE STAYS
# ============================================================

needed_lab_items = set()

for ids in lab_groups.values():
    needed_lab_items.update(ids)

print(
    "\nLab item IDs required:",
    len(needed_lab_items)
)

lab_parts = []

for chunk in pd.read_csv(
    HOSP / "labevents.csv.gz",
    compression="gzip",
    chunksize=500_000
):

    filtered = chunk[
        chunk["hadm_id"].isin(sae_hadm)
        &
        chunk["itemid"].isin(needed_lab_items)
    ].copy()

    if len(filtered) > 0:
        lab_parts.append(filtered)

if lab_parts:
    labs = pd.concat(
        lab_parts,
        ignore_index=True
    )
else:
    labs = pd.DataFrame()

print(
    "\nFiltered lab events:",
    labs.shape
)

# ============================================================
# 7. MAP LAB ITEMS
# ============================================================

if len(labs) > 0:

    lab_to_feature = {}

    for feature, ids in lab_groups.items():
        for itemid in ids:
            lab_to_feature[itemid] = feature

    labs["feature"] = labs["itemid"].map(
        lab_to_feature
    )

    labs["charttime"] = pd.to_datetime(
        labs["charttime"]
    )

    labs["valuenum"] = pd.to_numeric(
        labs["valuenum"],
        errors="coerce"
    )

    labs = labs.dropna(
        subset=["valuenum", "charttime"]
    )

    # Map labs to ICU stay
    labs = labs.merge(
        sae_icu[
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "intime",
                "outtime"
            ]
        ],
        on=["subject_id", "hadm_id"],
        how="inner"
    )

    labs = labs[
        (labs["charttime"] >= labs["intime"])
        &
        (labs["charttime"] <= labs["outtime"])
    ]

    labs["hour_time"] = (
        labs["charttime"]
        .dt.floor("h")
    )

    lab_hourly = (
        labs
        .groupby(
            [
                "subject_id",
                "hadm_id",
                "stay_id",
                "hour_time",
                "feature"
            ],
            as_index=False
        )["valuenum"]
        .median()
    )

    lab_hourly = lab_hourly.pivot_table(
        index=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        columns="feature",
        values="valuenum",
        aggfunc="first"
    ).reset_index()

else:

    lab_hourly = pd.DataFrame()

print(
    "\nHourly laboratory feature table:",
    lab_hourly.shape
)

# ============================================================
# 8. CREATE COMPLETE HOURLY GRID
# ============================================================

rows = []

for _, stay in sae_icu.iterrows():

    start = stay["intime"].floor("h")
    end = stay["outtime"].floor("h")

    hours = pd.date_range(
        start=start,
        end=end,
        freq="h"
    )

    temp = pd.DataFrame({
        "subject_id": stay["subject_id"],
        "hadm_id": stay["hadm_id"],
        "stay_id": stay["stay_id"],
        "hour_time": hours
    })

    rows.append(temp)

hourly = pd.concat(
    rows,
    ignore_index=True
)

print(
    "\nComplete hourly grid:",
    hourly.shape
)

# ============================================================
# 9. MERGE VITALS + NEURO + LABS
# ============================================================

if len(chart_hourly) > 0:

    hourly = hourly.merge(
        chart_hourly,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        how="left"
    )

if len(lab_hourly) > 0:

    hourly = hourly.merge(
        lab_hourly,
        on=[
            "subject_id",
            "hadm_id",
            "stay_id",
            "hour_time"
        ],
        how="left"
    )

# ============================================================
# 10. ADD MISSINGNESS INDICATORS
# ============================================================

feature_columns = [
    c for c in hourly.columns
    if c not in [
        "subject_id",
        "hadm_id",
        "stay_id",
        "hour_time"
    ]
]

for c in feature_columns:

    hourly[c + "_missing"] = (
        hourly[c].isna().astype(int)
    )

# ============================================================
# 11. FORWARD-FILL WITHIN EACH ICU STAY
# ============================================================

numeric_features = [
    c for c in feature_columns
    if pd.api.types.is_numeric_dtype(hourly[c])
]

hourly = hourly.sort_values(
    [
        "stay_id",
        "hour_time"
    ]
)

hourly[numeric_features] = (
    hourly
    .groupby("stay_id")[numeric_features]
    .ffill()
)

# ============================================================
# 12. ADD SAFE GLOBAL MEDIAN FALLBACK
# ============================================================

for c in numeric_features:

    median_value = hourly[c].median()

    if pd.notna(median_value):
        hourly[c] = hourly[c].fillna(
            median_value
        )

# ============================================================
# 13. ADD RELATIVE ICU HOUR
# ============================================================

icu_start = (
    hourly
    .groupby("stay_id")["hour_time"]
    .transform("min")
)

hourly["hour"] = (
    (
        hourly["hour_time"] - icu_start
    ).dt.total_seconds() / 3600
).astype(int)

# ============================================================
# 14. ADD SHORT-TERM DELTA FEATURES
# ============================================================

delta_features = [
    "heart_rate",
    "map",
    "resp_rate",
    "spo2",
    "temperature",
    "gcs_total",
    "rass",
    "lactate",
    "wbc",
    "creatinine",
    "bilirubin",
    "platelets",
    "bun",
    "sodium",
    "glucose",
    "ph"
]

for c in delta_features:

    if c in hourly.columns:

        hourly[c + "_delta"] = (
            hourly
            .groupby("stay_id")[c]
            .diff()
        )

# ============================================================
# 15. SAVE FEATURE TABLE
# ============================================================

output_file = OUT / "mimiciv_sae_hourly_features.csv"

hourly.to_csv(
    output_file,
    index=False
)

print("\n" + "=" * 60)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 60)

print(
    "Hourly rows:",
    len(hourly)
)

print(
    "Patients:",
    hourly["subject_id"].nunique()
)

print(
    "ICU stays:",
    hourly["stay_id"].nunique()
)

print(
    "Columns:",
    len(hourly.columns)
)

print(
    "\nFeature columns:"
)

print(
    [
        c for c in hourly.columns
        if not c.endswith("_missing")
    ]
)

print(
    "\nSaved:",
    output_file
)

print("\nMissing-value summary BEFORE any model training:")
print(
    hourly.isna().sum()
    .sort_values(ascending=False)
    .head(30)
)

Project: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning
MIMIC-IV: /Users/chinmay/Clg Stuff/Research Stuff/SAE_RL_EarlyWarning/data/raw/mimic-iv
SAE COHORT
SAE admissions: 6
SAE patients: 6
SAE ICU stays: 9

ICU item dictionary: (4014, 9)

heart_rate: 3 matching items
 itemid                   label
 220047  Heart Rate Alarm - Low
 220046 Heart rate Alarm - High
 220045              Heart Rate

resp_rate: 4 matching items
 itemid                          label
 224688         Respiratory Rate (Set)
 224690       Respiratory Rate (Total)
 220210               Respiratory Rate
 224689 Respiratory Rate (spontaneous)

spo2: 3 matching items
 itemid                         label
 228232         PAR-Oxygen saturation
 226253              SpO2 Desat Limit
 229862 Forehead SpO2 Sensor in Place

map: 0 matching items

temperature: 9 matching items
 itemid                       label
 224674      Changes in Temperature
 224027            Skin Temperature
 224642            Temperatur